In [1]:
# Imports and Constants
import pandas as pd
import numpy as np
import sqlite3
import joblib
import torch
from torch import Tensor

from simple_diffusion import Noiser, ClimbDDPM
from hold_classifier import UNetHoldClassifierLogits
from climb_conversion import ClimbsFeatureScaler

DB_PATH = "data/storage.db"
SCALER_WEIGHTS_PATH = 'data/weights/climbs-feature-scaler.joblib'
DDPM_WEIGHTS_PATH = 'data/weights/simple-diffusion-large.pth'
HC_WEIGHTS_PATH = 'data/weights/unet-hold-classifier.pth'
WALL_ID = 'wall-0a877f13d8e5'

GRADE_TO_DIFF = {
    "font": {
        "4a": 10, "4b": 11, "4c": 12,
        "5a": 13, "5b": 14, "5c": 15,
        "6a": 16, "6a+": 17, "6b": 18, "6b+": 19,
        "6c": 20, "6c+": 21,
        "7a": 22, "7a+": 23, "7b": 24, "7b+": 25,
        "7c": 26, "7c+": 27,
        "8a": 28, "8a+": 29, "8b": 30, "8b+": 31,
        "8c": 32, "8c+": 33,
    },
    "v_grade": {
        "V0-": 10, "V0": 11, "V0+": 12,
        "V1": 13, "V1+": 14, "V2": 15,
        "V3": 16, "V3+": 17, "V4": 18, "V4+": 19,
        "V5": 20, "V5+": 21, "V6": 22, "V6+": 22.5,
        "V7": 23, "V7+": 23.5, "V8": 24, "V8+": 25,
        "V9": 26, "V9+": 26.5, "V10": 27, "V10+": 27.5,
        "V11": 28, "V11+": 28.5, "V12": 29, "V12+": 29.5,
        "V13": 30, "V13+": 30.5, "V14": 31, "V14+": 31.5,
        "V15": 32, "V15+": 32.5, "V16": 33,
    },
}
NULL_HOLD_SENTINELS = torch.tensor(
    [[-2.0, 0.0, -2.0, 0.0],
    [2.0, 0.0, -2.0, 0.0],
    [-2.0, 0.0, 2.0, 0.0],
    [2.0, 0.0, 2.0, 0.0]], dtype=torch.float32)

In [8]:
# ---------------------------------------------------------------------------
# ClimbDDPMGenerator
# ---------------------------------------------------------------------------

class ClimbDDPMGenerator():
    def __init__(
            self,
            scaler: ClimbsFeatureScaler,
            ddpm: ClimbDDPM,
            hold_classifier: UNetHoldClassifierLogits,
        ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler = scaler
        self.ddpm = ddpm
        self.hold_classifier = hold_classifier
        self._cond_cache = {}
        self.deterministic_noise_generator = torch.Generator(device=self.device)

        with sqlite3.connect(DB_PATH) as conn:
            holds = pd.read_sql_query("SELECT hold_index, x, y, pull_x, pull_y, useability, is_foot, wall_id FROM holds", conn)
            wall_ids = list(set(holds['wall_id'].values))
            scaled_holds = self.scaler.transform_hold_features(holds, to_df=True)
            self.holds_manifolds = {}
            self.holds_lookup = {}
            for wall_id in wall_ids:
                df = scaled_holds[scaled_holds['wall_id']==wall_id]
                self.holds_manifolds[wall_id] = torch.cat([torch.tensor(df[['x','y','pull_x','pull_y']].values, dtype=torch.float32), NULL_HOLD_SENTINELS], dim=0)
                self.holds_lookup[wall_id] = df['hold_index'].values
                self.holds_lookup[wall_id] = np.concatenate([self.holds_lookup[wall_id], np.array([-1, -1, -1, -1])])

    def _build_cond_tensor(self, n, grade, diff_scale, angle):
        cache_key = (grade, diff_scale, angle)
        if cache_key not in self._cond_cache:
            diff = GRADE_TO_DIFF[diff_scale][grade]
            row = np.array([[diff, 3.0, 1000, float(angle)]])
            row[:, 1] = np.log(1 - (row[:, 1] - 3))     # quality
            row[:, 2] = np.log(row[:, 2])               # ascents
            scaled = self.scaler.cond_features_scaler.transform(row)
            self._cond_cache[cache_key] = scaled[0]
        base = self._cond_cache[cache_key]
        tiled = np.tile(base, (n,1))
        return torch.tensor(tiled, device=self.device, dtype=torch.float32)
    
    def _project_onto_manifold(self, gen_climbs: Tensor, offset_manifold: Tensor)-> Tensor:
        """
            Project each generated hold to its nearest neighbor on the hold manifold.
            
            Args:
                gen_climbs: (B, S, H) predicted clean holds
                return_indices: (boolean) Whether to return the hold indices or hold feature coordinates
            Returns:
                projected: (B, S, H) each hold snapped to nearest manifold point
        """
        B, S, H = gen_climbs.shape
        flat_climbs = gen_climbs.reshape(-1,H)
        dists = torch.cdist(flat_climbs, offset_manifold)
        idx = dists.argmin(dim=1)
        return offset_manifold[idx].reshape(B, S, -1)
    
    def _find_optimal_translation(
        self,
        gen_climbs: Tensor,           # (B, S, H)
        offset_manifold: Tensor,      # (M, H)
        x_offsets: Tensor,            # (Nx,)
        y_offsets: Tensor,            # (Ny,)
    ) -> tuple[Tensor, Tensor]:
        """
        Find the (dx, dy) translation per batch item that minimises total
        nearest-neighbour projection distance onto the hold manifold.

        Returns:
            best_dx: (B,)  optimal x translation for each climb
            best_dy: (B,)  optimal y translation for each climb
        """
        B, S, H = gen_climbs.shape

        # Create a null mask so that the null-hold positions don't contribute to the distance metric.
        null_mask = ((gen_climbs[:,:,0] < -1.2) | (gen_climbs[:,:,2] > 1.2 ) | (gen_climbs[:,:,0] > 1.2) | (gen_climbs[:,:,2] < -1.2)).float()
        Nx = x_offsets.shape[0]
        Ny = y_offsets.shape[0]

        # Build a (Nx*Ny, H) manifold shift table — only x/y cols (0 and 2) move
        # Shape: (Nx, Ny, H)
        shifts = torch.zeros(Nx, Ny, H, device=gen_climbs.device)
        shifts[:, :, 0] = x_offsets.unsqueeze(1)   # broadcast over Ny
        shifts[:, :, 1] = y_offsets.unsqueeze(0)   # broadcast over Nx
        G = Nx * Ny
        shifts = shifts.reshape(G, H)         # (G, H)  G = grid size

        # Translate climbs: (B, S, H) + (G, H) -> (G, B, S, H)
        translated = gen_climbs.unsqueeze(0) + shifts.unsqueeze(1).unsqueeze(2)

        # Flatten holds dim for cdist: (G*B, S, H)
        flat = translated.reshape(G * B, S, H)

        # Nearest-neighbour distances to manifold: (G*B, S, M) -> (G*B, S)
        dists = torch.cdist(flat, offset_manifold)              # (G*B, S, M)
        nn_dists = dists.min(dim=2).values                      # (G*B, S)
        batch_dist = nn_dists.reshape(G, B, S)                  # (G, B, S)
        batch_dist = null_mask.unsqueeze(0) * batch_dist        # (G, B, S)

        total_dist = batch_dist.sum(dim=2)

        # Best grid point per batch item
        best_g = total_dist.argmin(dim=0)                       # (B,)
        best_dx = x_offsets[best_g // Ny]
        best_dy = y_offsets[best_g % Ny]

        return best_dx, best_dy

    def _project_onto_indices_with_translation(
        self,
        gen_climbs: Tensor,
        cond_t: Tensor,
        offset_manifold: Tensor,
        wall_id: str,
        x_offsets: Tensor | None = None,
        y_offsets: Tensor | None = None,
    ) -> list[list[list[int]]]:

        if x_offsets is None:
            x_offsets = torch.linspace(-0.5, 0.5, 51, device=gen_climbs.device)
        if y_offsets is None:
            y_offsets = torch.linspace(-0.25, 0.25, 51, device=gen_climbs.device)

        best_dx, best_dy = self._find_optimal_translation(
            gen_climbs, offset_manifold, x_offsets, y_offsets
        )
        print(f"Best Dx:{best_dx}, Best Dy: {best_dy}")

        # Apply per-climb optimal translation  (B, S, H)
        B, S, H = gen_climbs.shape
        translation = torch.zeros(B, 1, H, device=gen_climbs.device)
        translation[:, 0, 0] = best_dx   # x col
        translation[:, 0, 1] = best_dy   # y col  (pull_x/pull_y cols 1,3 left alone)
        translated_climbs = gen_climbs + translation

        # Now do the standard index projection on the translated climbs
        return self._project_onto_indices(translated_climbs, cond_t, offset_manifold, wall_id)

    def _project_onto_indices(self, gen_climbs: Tensor, cond_t: Tensor, offset_manifold: Tensor, wall_id: str) -> list[list[list[int]]]:
        """Project climb onto the final hold indices (and remove null holds)"""
        
        B, S, H = gen_climbs.shape

        roles = torch.argmax(self.hold_classifier(gen_climbs, cond_t), dim=2).detach().numpy()

        flat_climbs = gen_climbs.reshape(-1,H)
        dists = torch.cdist(flat_climbs, offset_manifold)
        idx = dists.argmin(dim=1)
        holds = self.holds_lookup[wall_id][idx]
        holds = holds.reshape(B, S)
        
        is_null = (holds == -1)
        roles[is_null] = 4
        
        climbs = np.stack([holds, roles], axis=2)
        
        deduped_climbs = []
        for c in climbs:
            valid_mask = c[:, 1] != 4
            c_valid = c[valid_mask]
            c_sorted = c_valid[c_valid[:, 1].argsort()]
            _, unique_indices = np.unique(c_sorted[:, 0], return_index=True)
            deduped_climbs.append(c_sorted[unique_indices].tolist())

        return deduped_climbs
    
    def _projection_strength(self, t: Tensor, t_start_projection: float):
        """Calculate the weight to assign to the projected holds based on the timestep."""
        a = (t_start_projection-t)/t_start_projection
        strength = 1 - torch.cos(a*torch.pi/2)
        return torch.where(t > t_start_projection, torch.zeros_like(t), strength).unsqueeze(2)
    
    @torch.no_grad()
    def generate(
        self,
        wall_id: str,
        n: int,
        angle: int,
        grade: str,
        diff_scale: str,
        timesteps: int,
        deterministic: bool,
        t_start_projection: float,
        x_offset: float | None,
        seed: int,
    )->list[list[list[int]]]:
        """
        Generate a climb or batch of climbs with the given conditions using the standard DDPM iterative denoising process.

        :param wall_id: The Wall-ID on which to generate the climb.
        :type wall_id: str
        :param n: The number of climbs to generate.
        :type n: int
        :param angle: The current wall angle.
        :type angle: int
        :param grade: The desired grade.
        :type grade: str
        :param diff_scale: The desired difficulty scale (V-scale or Font).
        :type diff_scale: str
        :param timesteps: Model setting: Number of diffusion timesteps to run. Higher results in better quality (Should be a divisor of 100 to retain markovian properties)
        :type timesteps: int
        :param deterministic: Whether to use the original noise vector in successive diffusion steps, or use a new noise vector each time.
        :type deterministic: bool
        :param t_start_projection: Point in the generation process to begin the projection steps. Earlier is better but more expensive.
        :type t_start_projection: float
        :param x_offset: Offset the climb on the X-axis.
        :type x_offset: float | None
        :return: A set of generated climbs according to the specified 
        :rtype: list[list[list[int]]]
        """
        auto=False
        if x_offset is None:
            auto=True
            x_offset=0
        
        # Seed Noise Generator
        if deterministic:
            self.deterministic_noise_generator.manual_seed(seed)
        
        offset_manifold = self.holds_manifolds[wall_id].clone()
        offset_manifold[:,0] += x_offset*0.1

        # CORE LOGIC
        cond_t = self._build_cond_tensor(n, grade, diff_scale, angle)
        x_t = torch.randn((n, 20, 4), device=self.device, generator=self.deterministic_noise_generator) if deterministic else torch.randn((n, 20, 4), device=self.device)
        noisy = x_t.clone()
        t_tensor = torch.ones((n,1), device=self.device)
        
        for _ in range(0, timesteps):
            gen_climbs = self.ddpm(noisy, cond_t, t_tensor)

            alpha_p = self._projection_strength(t_tensor, t_start_projection)
            projected_climbs = self._project_onto_manifold(gen_climbs, offset_manifold)
            gen_climbs = alpha_p*(projected_climbs) + (1-alpha_p)*(gen_climbs)
            
            t_tensor -= 1.0/timesteps
            noisy = self.ddpm.forward_diffusion(gen_climbs, t_tensor, x_t if deterministic else torch.randn_like(x_t))
        
        if auto:
            return self._project_onto_indices_with_translation(gen_climbs, cond_t, offset_manifold, wall_id)
        return self._project_onto_indices(gen_climbs, cond_t, offset_manifold, wall_id)

# ---------------------------------------------------------------------------
# Global ClimbGenerator Instance For Dependency Injection
# ---------------------------------------------------------------------------
scaler = ClimbsFeatureScaler(
    weights_path=SCALER_WEIGHTS_PATH
)
ddpm = ClimbDDPM(
    model=Noiser(),
    weights_path=DDPM_WEIGHTS_PATH,
)
hold_classifier = UNetHoldClassifierLogits(
    weights_path=HC_WEIGHTS_PATH
)

# Set to Eval() and compile
ddpm.eval()
# ddpm = torch.compile(ddpm)
hold_classifier.eval()
# hold_classifier = torch.compile(hold_classifier)

generator = ClimbDDPMGenerator(
    scaler=scaler,
    ddpm=ddpm,
    hold_classifier=hold_classifier
)

for _ in range(3):
    generator.generate(
        "wall-9e000814e880", 10, 45, "V4", "v_grade", 100, False, 0.0, None, 37
    )

c:\Users\EvanM\Documents\Projects\GitHub\ml-homewall-climb-generator\model-training\simple_diffusion.py:611: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch

Best Dx:tensor([-0.0200, -0.0200, -0.0200,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
        -0.0200,  0.0000]), Best Dy: tensor([0.0100, 0.0000, 0.0100, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0100,
        0.0000])
Best Dx:tensor([ 0.0000,  0.0000, -0.0200,  0.0000,  0.0000, -0.0200,  0.0000,  0.0000,
         0.0000,  0.0000]), Best Dy: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0100, 0.0100, 0.0000, 0.0000, 0.0000,
        0.0000])
Best Dx:tensor([ 0.0000,  0.0000,  0.0000, -0.0200, -0.0200,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000]), Best Dy: tensor([0.0000, 0.0100, 0.0100, 0.0100, 0.0000, 0.0000, 0.0000, 0.0000, 0.0100,
        0.0000])
